## Project title and overview

In [ ]:
import os
import glob
import hashlib
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

np.random.seed(42)
tf.random.set_seed(42)

DATASET_DIR = "../digit_dataset"
IMG_SIZE = 48  # bumped from 32: more detail helps separate 3/5/8/2
CLASSES = [str(d) for d in range(10)]

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)


## Step 2 — Verify Dataset

In [ ]:

print("Current working directory:")
print(os.getcwd())

print("\nDataset path:")
print(os.path.abspath(DATASET_DIR))

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(
        f"\nDataset folder was not found:\n{os.path.abspath(DATASET_DIR)}\n\n"
        "Make sure 'digit_dataset' is in the correct location."
    )

print("\nDataset folder found successfully! ✅")

print("\nDataset contents:")
print(sorted(os.listdir(DATASET_DIR)))

## Step 3 — Verify Class Folders

In [ ]:

missing_classes = []

for cls in CLASSES:
    folder = os.path.join(DATASET_DIR, cls)

    if os.path.isdir(folder):
        print(f"Digit {cls}: ✅ folder exists")
    else:
        print(f"Digit {cls}: ❌ folder missing")
        missing_classes.append(cls)

if missing_classes:
    raise FileNotFoundError(
        f"\nMissing digit folders: {missing_classes}"
    )

print("\nAll 10 digit folders are present! ✅")

## Step 4 — Dataset Scan

In [ ]:

def list_image_files(folder):
    """
    Lists each image file exactly once. Needed because on
    case-insensitive filesystems (Windows), glob.glob("*.jpg")
    and glob.glob("*.JPG") match the SAME files, so the old
    code was counting (and later loading) every image twice.
    """
    if not os.path.isdir(folder):
        return []
    files = []
    seen = set()
    for name in os.listdir(folder):
        if name.lower().endswith((".jpg", ".jpeg", ".png")):
            full_path = os.path.join(folder, name)
            key = os.path.normcase(os.path.abspath(full_path))
            if key not in seen:
                seen.add(key)
                files.append(full_path)
    return sorted(files)


def scan_dataset(dataset_dir):
    report = {
        "per_class_count": {},
        "corrupt_files": [],
        "resolutions": [],
        "total_files": 0
    }
    for cls in CLASSES:
        folder = os.path.join(dataset_dir, cls)
        files = list_image_files(folder)

        report["per_class_count"][cls] = len(files)
        report["total_files"] += len(files)

        for file_path in files:
            img = cv2.imread(file_path)
            if img is None:
                report["corrupt_files"].append(file_path)
            else:
                report["resolutions"].append(img.shape[:2])
    return report


report = scan_dataset(DATASET_DIR)

print("Images per class:")
print("-" * 30)
for cls, count in report["per_class_count"].items():
    print(f"Digit {cls}: {count}")
print("-" * 30)
print("Total images:", report["total_files"])
print("Corrupt/unreadable images:", len(report["corrupt_files"]))

if report["corrupt_files"]:
    print("\nCorrupt files:")
    for file_path in report["corrupt_files"]:
        print(" ", file_path)

if report["resolutions"]:
    resolutions = np.array(report["resolutions"])
    print("\nResolution statistics:")
    print("Minimum (H, W):", resolutions.min(axis=0))
    print("Maximum (H, W):", resolutions.max(axis=0))
    print("Mean (H, W):", resolutions.mean(axis=0).round(1))

## Step 5 — Check Dataset Requirement

In [ ]:

print("Checking the instructor's requirement:")
print("Requirement: 50–100 images for EACH digit\n")

requirement_ok = True

for cls, count in report["per_class_count"].items():

    if 50 <= count <= 200:
        status = "✅"
    else:
        status = "❌"
        requirement_ok = False

    print(f"Digit {cls}: {count:3d} images {status}")

print("\nTotal images:", report["total_files"])

if requirement_ok:
    print("\n✅ Every digit satisfies the 50–100 image requirement.")
else:
    print("\n⚠️ At least one digit is outside the required range.")

## Step 6 — Visualize Raw Images

In [ ]:
import random
def show_raw_samples(dataset_dir, n_per_class=5, seed=42):

    rng = random.Random(seed)

    fig, axes = plt.subplots(
        len(CLASSES),
        n_per_class,
        figsize=(12, 20)
    )

    for row, cls in enumerate(CLASSES):

        files = (
            glob.glob(os.path.join(dataset_dir, cls, "*.jpg")) +
            glob.glob(os.path.join(dataset_dir, cls, "*.JPG"))
        )

        rng.shuffle(files)
        files = files[:n_per_class]

        for col in range(n_per_class):

            ax = axes[row, col]
            ax.axis("off")

            if col < len(files):

                img = cv2.imread(files[col])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                ax.imshow(img)

            if col == 0:
                ax.set_ylabel(
                    f"Digit {cls}",
                    rotation=0,
                    labelpad=35,
                    fontsize=11
                )

    plt.suptitle(
        "Random Raw Samples from Each Digit",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


show_raw_samples(DATASET_DIR, n_per_class=5)

## Step 7 — Detect Suspicious Images

In [ ]:

def laplacian_blur_score(img_gray):
    """
    Higher score generally means a sharper image.
    This is only a screening tool, not an automatic
    decision that an image is bad.
    """
    return cv2.Laplacian(
        img_gray,
        cv2.CV_64F
    ).var()


def find_suspicious_images(dataset_dir, blur_threshold=50.0):

    blurry = []
    duplicates = []

    seen_hashes = {}

    for cls in CLASSES:

        files = (
            glob.glob(os.path.join(dataset_dir, cls, "*.jpg")) +
            glob.glob(os.path.join(dataset_dir, cls, "*.JPG"))
        )

        for file_path in files:

            img = cv2.imread(file_path)

            if img is None:
                continue

            gray = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2GRAY
            )

            score = laplacian_blur_score(gray)

            if score < blur_threshold:
                blurry.append(
                    (file_path, round(score, 2))
                )

            # Hash file for exact duplicate detection
            with open(file_path, "rb") as f:
                file_hash = hashlib.md5(
                    f.read()
                ).hexdigest()

            if file_hash in seen_hashes:
                duplicates.append(
                    (file_path, seen_hashes[file_hash])
                )
            else:
                seen_hashes[file_hash] = file_path

    return blurry, duplicates


blurry, duplicates = find_suspicious_images(
    DATASET_DIR,
    blur_threshold=50.0
)

print(f"Potentially blurry images: {len(blurry)}")

for file_path, score in blurry[:30]:
    print(f"{score:8.2f}  {file_path}")

if len(blurry) > 30:
    print(f"... and {len(blurry) - 30} more")


print("\nExact duplicate files:", len(duplicates))

for duplicate, original in duplicates[:30]:
    print(f"{duplicate}")
    print(f"  duplicate of → {original}")

## Step 8 — Preprocessing Functions

In [ ]:

def find_paper_region(gray):
    """
    Try to locate the paper/background region.
    Returns (x0, y0, x1, y1) or None.
    """

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    _, mask = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    largest = max(
        contours,
        key=cv2.contourArea
    )

    image_area = gray.shape[0] * gray.shape[1]

    if cv2.contourArea(largest) < image_area * 0.15:
        return None

    x, y, w, h = cv2.boundingRect(largest)

    # Small inward margin
    sx = int(w * 0.05)
    sy = int(h * 0.05)

    x0 = x + sx
    y0 = y + sy
    x1 = x + w - sx
    y1 = y + h - sy

    return x0, y0, x1, y1


def crop_digit(img_bgr, pad=15):

    gray_full = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2GRAY
    )

    # ----------------------------------------
    # Stage 1: Find paper
    # ----------------------------------------

    paper_box = find_paper_region(gray_full)

    if paper_box is not None:

        x0, y0, x1, y1 = paper_box

        paper_bgr = img_bgr[
            y0:y1,
            x0:x1
        ]

        paper_gray = gray_full[
            y0:y1,
            x0:x1
        ]

    else:

        paper_bgr = img_bgr
        paper_gray = gray_full

    # ----------------------------------------
    # Stage 2: Find digit
    # ----------------------------------------

    blur = cv2.GaussianBlur(
        paper_gray,
        (5, 5),
        0
    )

    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        35,
        10
    )

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    thresh = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    thresh = cv2.dilate(
        thresh,
        kernel,
        iterations=2
    )

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Remove very small noise
    contours = [
        c for c in contours
        if cv2.contourArea(c) > 20
    ]

    if not contours:
        return paper_bgr

    xs, ys, xe, ye = [], [], [], []

    for c in contours:

        x, y, w, h = cv2.boundingRect(c)

        xs.append(x)
        ys.append(y)
        xe.append(x + w)
        ye.append(y + h)

    x0d = max(min(xs) - pad, 0)
    y0d = max(min(ys) - pad, 0)

    x1d = min(
        max(xe) + pad,
        paper_bgr.shape[1]
    )

    y1d = min(
        max(ye) + pad,
        paper_bgr.shape[0]
    )

    return paper_bgr[
        y0d:y1d,
        x0d:x1d
    ]


def normalize_lighting(gray_img):

    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )

    return clahe.apply(gray_img)


def center_digit(gray_img):

    blur = cv2.GaussianBlur(
        gray_img,
        (5, 5),
        0
    )

    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        35,
        10
    )

    M = cv2.moments(thresh)

    if M["m00"] == 0:
        return gray_img

    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]

    h, w = gray_img.shape

    shift_x = w / 2 - cx
    shift_y = h / 2 - cy

    transformation = np.float32([
        [1, 0, shift_x],
        [0, 1, shift_y]
    ])

    centered = cv2.warpAffine(
        gray_img,
        transformation,
        (w, h),
        borderValue=255
    )

    return centered

## Step 9 — Complete Preprocessing Pipeline

In [ ]:

def preprocess_image(
    path,
    img_size=IMG_SIZE
):
    """
    Preprocessing pipeline:

    Raw image
        ↓
    Crop digit
        ↓
    Grayscale
        ↓
    Lighting normalization
        ↓
    Center digit
        ↓
    Resize to 32x32
        ↓
    Normalize to [0,1]
    """

    img = cv2.imread(path)

    if img is None:
        return None

    # 1. Crop
    cropped = crop_digit(img)

    # 2. Grayscale
    gray = cv2.cvtColor(
        cropped,
        cv2.COLOR_BGR2GRAY
    )

    # 3. Lighting/background normalization
    normalized_lighting = normalize_lighting(gray)

    # 4. Center
    centered = center_digit(
        normalized_lighting
    )

    # 5. Resize
    resized = cv2.resize(
        centered,
        (img_size, img_size),
        interpolation=cv2.INTER_AREA
    )

    # 6. Normalize [0,1]
    normalized = (
        resized.astype("float32") / 255.0
    )

    return normalized

## Step 10 — Inspect Preprocessing

In [ ]:
def show_preprocessing_examples(
    dataset_dir,
    n_per_class=3,
    seed=42
):

    rng = random.Random(seed)

    fig, axes = plt.subplots(
        len(CLASSES),
        n_per_class * 2,
        figsize=(10, 20)
    )

    for row, cls in enumerate(CLASSES):

        files = (
            glob.glob(
                os.path.join(
                    dataset_dir,
                    cls,
                    "*.jpg"
                )
            )
            +
            glob.glob(
                os.path.join(
                    dataset_dir,
                    cls,
                    "*.JPG"
                )
            )
        )

        rng.shuffle(files)

        files = files[:n_per_class]

        for col in range(n_per_class):

            if col >= len(files):
                continue

            path = files[col]

            # Raw
            raw = cv2.imread(path)
            raw = cv2.cvtColor(
                raw,
                cv2.COLOR_BGR2RGB
            )

            # Processed
            processed = preprocess_image(path)

            raw_ax = axes[row, col * 2]
            processed_ax = axes[row, col * 2 + 1]

            raw_ax.imshow(raw)
            raw_ax.axis("off")
            raw_ax.set_title("Raw")

            processed_ax.imshow(
                processed,
                cmap="gray",
                vmin=0,
                vmax=1
            )
            processed_ax.axis("off")
            processed_ax.set_title("Processed")

        axes[row, 0].set_ylabel(
            f"Digit {cls}",
            rotation=0,
            labelpad=35
        )

    plt.suptitle(
        "Raw vs Preprocessed Images",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


show_preprocessing_examples(
    DATASET_DIR,
    n_per_class=3
)

## Step 11 — Build X and y

In [ ]:
X = []
y = []
skipped = []

for cls in CLASSES:

    files = (
        glob.glob(
            os.path.join(
                DATASET_DIR,
                cls,
                "*.jpg"
            )
        )
        +
        glob.glob(
            os.path.join(
                DATASET_DIR,
                cls,
                "*.JPG"
            )
        )
    )

    files = sorted(files)

    for file_path in files:

        processed = preprocess_image(file_path)

        if processed is None:
            skipped.append(file_path)
            continue

        X.append(processed)
        y.append(int(cls))


X = np.array(
    X,
    dtype="float32"
)

y = np.array(
    y,
    dtype="int64"
)

# Add CNN channel dimension
X = X.reshape(
    -1,
    IMG_SIZE,
    IMG_SIZE,
    1
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Skipped:", len(skipped))

print("\nPixel range:")
print("Minimum:", X.min())
print("Maximum:", X.max())

In [ ]:
# ============================================================
# NEW: sanity-check for preprocessing failures
# ------------------------------------------------------------
# This is the single most important cell for debugging the
# "predicts everything as the same digit" problem. If cropping
# fails on a photo, the resulting 32x32 image ends up almost
# entirely white/blank (ink ratio ~ 0). If MANY images across
# your classes come back near-blank, that is direct proof the
# preprocessing pipeline -- not the CNN -- is the problem, and
# you should fix/relax crop_digit() or retake those photos
# (better lighting, digit written large and dark, plain
# background) before training.
# ============================================================

flat = X.reshape(len(X), -1)          # X is already background=~1 (white), ink=~0
ink_ratio = 1.0 - flat.mean(axis=1)   # higher = more dark ink pixels

print("Ink-ratio stats across the whole processed dataset:")
print(f"  min : {ink_ratio.min():.4f}")
print(f"  mean: {ink_ratio.mean():.4f}")
print(f"  max : {ink_ratio.max():.4f}")

BLANK_THRESHOLD = 0.015   # tune if needed after looking at a histogram
blank_idx = np.where(ink_ratio < BLANK_THRESHOLD)[0]

print(f"\nNear-blank images (likely crop failures): {len(blank_idx)} / {len(X)}")

if len(blank_idx) > 0:
    print("Per-digit breakdown of near-blank images:")
    for digit in range(10):
        n = int(np.sum(y[blank_idx] == digit))
        if n:
            print(f"  Digit {digit}: {n}")

    n_show = min(10, len(blank_idx))
    fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2))
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, blank_idx[:n_show]):
        ax.imshow(X[idx].squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"y={y[idx]}")
        ax.axis("off")
    plt.suptitle("Examples flagged as near-blank (crop probably failed)")
    plt.tight_layout()
    plt.show()

    print(
        "\n⚠️ If this list is non-trivial, do NOT train yet. Either "
        "retake these photos (dark ink, plain background, good "
        "lighting) or loosen crop_digit()'s min_area_frac, then "
        "re-run preprocessing from the 'X = []' cell above."
    )
else:
    print("✅ No near-blank images detected -- preprocessing looks healthy.")

plt.figure(figsize=(6, 3))
plt.hist(ink_ratio, bins=30)
plt.axvline(BLANK_THRESHOLD, color="red", linestyle="--", label="blank threshold")
plt.xlabel("Ink ratio")
plt.ylabel("Count")
plt.title("Distribution of ink ratio across dataset")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Dataset configuration
NUM_CLASSES = 10
CLASSES = [str(d) for d in range(NUM_CLASSES)]

## Step 12 — Verify Processed Dataset

In [ ]:
NUM_CLASSES = 10

print("Processed class distribution:")
print("-" * 30)

for digit in range(NUM_CLASSES):

    count = int(
        np.sum(y == digit)
    )

    print(
        f"Digit {digit}: {count}"
    )

print("-" * 30)
print("Total:", len(y))

## Step 13 — Save Preprocessed Dataset

In [ ]:
PREPROCESSED_FILE = (
    "digit_dataset_preprocessed.npz"
)

np.savez_compressed(
    PREPROCESSED_FILE,
    X=X,
    y=y
)

print(
    f"Saved preprocessed dataset to: "
    f"{PREPROCESSED_FILE}"
)

## Step 14 — Train / Validation / Test Spli

In [ ]:
SEED = 42

# First: separate 15% test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=SEED
)

# Second: take 15% of the total as validation
# Since 15% of the remaining 85% ≈ 12.75% total,
# use 0.1765 to obtain approximately 15% total validation.
validation_fraction = 0.15 / 0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_fraction,
    stratify=y_train_val,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nApproximate percentages:")
total = len(X)

print(
    f"Train: {len(X_train)/total:.2%}"
)

print(
    f"Validation: {len(X_val)/total:.2%}"
)

print(
    f"Test: {len(X_test)/total:.2%}"
)

## Step 15 — Class Distribution

In [ ]:

def get_class_counts(labels):

    return [
        int(np.sum(labels == digit))
        for digit in range(NUM_CLASSES)
    ]


train_counts = get_class_counts(y_train)
val_counts = get_class_counts(y_val)
test_counts = get_class_counts(y_test)

x = np.arange(NUM_CLASSES)
width = 0.25

plt.figure(figsize=(10, 5))

plt.bar(
    x - width,
    train_counts,
    width,
    label="Train"
)

plt.bar(
    x,
    val_counts,
    width,
    label="Validation"
)

plt.bar(
    x + width,
    test_counts,
    width,
    label="Test"
)

plt.xticks(
    x,
    CLASSES
)

plt.xlabel("Digit")
plt.ylabel("Number of Images")
plt.title("Class Distribution Across Dataset Splits")
plt.legend()

plt.tight_layout()
plt.show()

print("Train:", train_counts)
print("Validation:", val_counts)
print("Test:", test_counts)

## Step 16 — Test Set Protection

In [ ]:
assert (
    len(X_train)
    + len(X_val)
    + len(X_test)
    == len(X)
)

print(
    "✅ Train + Validation + Test "
    "contains all original samples."
)

print(
    "✅ Test set has been separated "
    "before augmentation."
)

print(
    "⚠️ Do not use X_test/y_test for "
    "model tuning."
)

## Step 17 — Training Augmentation

In [ ]:
# Slightly less aggressive than before -- large rotation/shift/shear can
# push a 2 toward looking like a 7, or a 6 toward a 5, which actively
# hurts your hardest classes. Keep just enough variety to generalize.
train_datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.10,
    shear_range=4,
    fill_mode="constant",
    cval=1.0
)

BATCH_SIZE = 32

train_generator = train_datagen.flow(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

print("Training augmentation created. ✅")
print("Validation data: NOT augmented.")
print("Test data: NOT augmented.")


## Step 18 — Augmentation Preview

In [ ]:
augmented_images, augmented_labels = next(
    train_generator
)

fig, axes = plt.subplots(
    2,
    5,
    figsize=(10, 4)
)

for i, ax in enumerate(axes.flat):

    ax.imshow(
        augmented_images[i].squeeze(),
        cmap="gray",
        vmin=0,
        vmax=1
    )

    ax.set_title(
        f"Label: {augmented_labels[i]}"
    )

    ax.axis("off")

plt.suptitle(
    "Examples of Training Data Augmentation"
)

plt.tight_layout()
plt.show()

## Step 19 — Small CNN

In [ ]:
def build_small_cnn(
    input_shape=(48, 48, 1),
    num_classes=10,
    dropout=0.4,
    filters=(16, 32, 64),
    learning_rate=1e-3
):
    # Added a third conv block now that IMG_SIZE=48 gives more
    # spatial detail to work with -- this is what lets the network
    # tell apart look-alike digits like 3/5/8 and 2/7, which a
    # 2-block network at 32x32 doesn't have enough resolution or
    # capacity for.

    model = models.Sequential([

        layers.Input(shape=input_shape),

        layers.Conv2D(filters[0], (3, 3), activation="relu", padding="same"),
        layers.SpatialDropout2D(0.1),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(filters[1], (3, 3), activation="relu", padding="same"),
        layers.SpatialDropout2D(0.1),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(filters[2], (3, 3), activation="relu", padding="same"),
        layers.SpatialDropout2D(0.15),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_small_cnn()
model.summary()


## Step 20 — Train Baseline CNN

In [ ]:
baseline_callbacks = [

    EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=15,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        "baseline_digit_cnn.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6
    )
]

EPOCHS = 100

history = model.fit(
    train_generator,

    steps_per_epoch=int(np.ceil(len(X_train) / BATCH_SIZE)),

    validation_data=(X_val, y_val),

    epochs=EPOCHS,

    callbacks=baseline_callbacks,

    verbose=1
)


## Step 21 — Training History

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 22 — CNN Model Comparison

In [ ]:
configs = [
    {
        "name": "CNN_A",
        "dropout": 0.3,
        "filters": (16, 32, 64),
        "lr": 1e-3
    },

    {
        "name": "CNN_B",
        "dropout": 0.5,
        "filters": (32, 64, 128),
        "lr": 1e-3
    },

    {
        "name": "CNN_C",
        "dropout": 0.4,
        "filters": (16, 32, 64),
        "lr": 5e-4
    }
]

variant_results = []

best_model = None
best_val_accuracy = -np.inf
best_config = None

for config in configs:

    print("\n" + "=" * 60)
    print("Training:", config["name"])
    print("=" * 60)

    variant_model = build_small_cnn(
        dropout=config["dropout"],
        filters=config["filters"],
        learning_rate=config["lr"]
    )

    variant_callbacks = [
        EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=12,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )
    ]

    variant_history = variant_model.fit(
        train_datagen.flow(
            X_train,
            y_train,
            batch_size=BATCH_SIZE,
            shuffle=True,
            seed=SEED
        ),

        steps_per_epoch=int(np.ceil(len(X_train) / BATCH_SIZE)),

        validation_data=(X_val, y_val),

        epochs=80,

        callbacks=variant_callbacks,

        verbose=0
    )

    current_best = max(variant_history.history["val_accuracy"])

    variant_results.append({
        "name": config["name"],
        "config": config,
        "best_val_accuracy": current_best
    })

    print(f"Best validation accuracy: {current_best:.4f}")

    if current_best > best_val_accuracy:
        best_val_accuracy = current_best
        best_model = variant_model
        best_config = config


print("\n" + "=" * 60)
print("BEST MODEL")
print("=" * 60)

print("Model:", best_config["name"])
print("Configuration:", best_config)
print("Best validation accuracy:", round(best_val_accuracy, 4))


## Step 23 — Variant Results

In [ ]:

for result in variant_results:

    print(
        f"{result['name']}: "
        f"{result['best_val_accuracy']:.4f}"
    )

## Step 24 — FINAL TEST EVALUATION

In [ ]:
final_model = best_model

test_loss, test_accuracy = final_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(
    f"Final Test Accuracy: "
    f"{test_accuracy:.4f}"
)

print(
    f"Final Test Loss: "
    f"{test_loss:.4f}"
)

## Step 25 — Predictions

In [ ]:
y_pred_probs = final_model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

print("Predictions generated successfully. ✅")

## Step 26 — Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=CLASSES,
        digits=4
    )
)

## Step 27 — Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES
)

plt.xlabel("Predicted Digit")
plt.ylabel("True Digit")
plt.title("Confusion Matrix — Test Set")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Which digit PAIRS are being confused the most?
# ------------------------------------------------------------
# Use this to decide where to focus: retake photos for those
# specific digits, or manually inspect/relabel a few samples --
# in handwritten datasets it's common for a handful of images
# to be genuinely ambiguous or mislabeled.
# ============================================================

pair_confusions = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            pair_confusions.append((cm[i, j], i, j))

pair_confusions.sort(reverse=True)

print("Top confused digit pairs (true -> predicted : count):")
for count, true_d, pred_d in pair_confusions[:10]:
    print(f"  {true_d} -> {pred_d} : {count}")


## Step 28 — Misclassified Images

In [ ]:
wrong_indices = np.where(
    y_pred != y_test
)[0]

print(
    f"Misclassified images: "
    f"{len(wrong_indices)} / {len(y_test)}"
)

if len(wrong_indices) > 0:

    n_show = min(
        12,
        len(wrong_indices)
    )

    fig, axes = plt.subplots(
        2,
        6,
        figsize=(14, 5)
    )

    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(
        axes,
        wrong_indices[:n_show]
    ):

        ax.imshow(
            X_test[idx].squeeze(),
            cmap="gray",
            vmin=0,
            vmax=1
        )

        ax.set_title(
            f"True: {y_test[idx]} | "
            f"Pred: {y_pred[idx]}"
        )

        ax.axis("off")

    plt.suptitle(
        "Misclassified Test Images"
    )

    plt.tight_layout()
    plt.show()

else:

    print(
        "No misclassified test images. Excellent!"
    )

## Step 29 — Save Best Model

In [ ]:
MODEL_PATH = "digit_recognition_model.keras"

final_model.save(
    MODEL_PATH
)

print(
    f"Best model saved to: {MODEL_PATH}"
)

## Step 30 — Reload Saved Model

In [ ]:

loaded_model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Saved model loaded successfully. ✅")

## Step 31 — Verify Loaded Model

In [ ]:
loaded_test_loss, loaded_test_accuracy = (
    loaded_model.evaluate(
        X_test,
        y_test,
        verbose=0
    )
)

print(
    f"Loaded model test accuracy: "
    f"{loaded_test_accuracy:.4f}"
)

## Step 32 — Prediction Function

In [ ]:
def predict_digit_from_path(
    path,
    model=loaded_model
):

    processed = preprocess_image(
        path
    )

    if processed is None:

        print(
            "Could not read image:"
        )

        print(path)

        return None

    # Add batch + channel dimensions
    batch = processed.reshape(
        1,
        IMG_SIZE,
        IMG_SIZE,
        1
    )

    probabilities = model.predict(
        batch,
        verbose=0
    )[0]

    prediction = int(
        np.argmax(probabilities)
    )

    confidence = float(
        probabilities[prediction]
    )

    plt.figure(figsize=(3, 3))

    plt.imshow(
        processed,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    plt.title(
        f"Predicted: {prediction}\n"
        f"Confidence: {confidence:.2%}"
    )

    plt.axis("off")
    plt.show()

    return prediction, probabilities

## Step 33 — Upload Prediction Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Digit"
)

output = widgets.Output()


def on_upload_change(change):

    with output:

        clear_output()

        if not upload_widget.value:
            return

        # ipywidgets 8
        if isinstance(
            upload_widget.value,
            dict
        ):

            item = list(
                upload_widget.value.values()
            )[0]

        # Older ipywidgets
        else:

            item = upload_widget.value[0]

        content = item["content"]

        temp_path = (
            "uploaded_digit.jpg"
        )

        with open(
            temp_path,
            "wb"
        ) as file:

            file.write(content)

        predict_digit_from_path(
            temp_path
        )


upload_widget.observe(
    on_upload_change,
    names="value"
)

display(
    upload_widget,
    output
)

## Step 34 — Deployment Sanity Test

In [ ]:
sample_test_files = (
    glob.glob(
        os.path.join(
            DATASET_DIR,
            "5",
            "*.jpg"
        )
    )
)

if sample_test_files:

    sample_test_path = (
        sample_test_files[0]
    )

    print(
        "Testing with:",
        sample_test_path
    )

    predict_digit_from_path(
        sample_test_path
    )

else:

    print(
        "No test image found."
    )